# AccessApp: On-Device ML Verification (MediaPipe)

This Google Colab notebook is provided to verify the core Machine Learning models used in **AccessApp**, satisfying the hackathon ML domain requirements.

Because our application runs exclusively on-device (Zero-Cloud architecture) using Edge TPU optimized models, this notebook replicates the **Obstacle Radar (Object Detection)** logic using the Python equivalent of our Android MediaPipe pipeline. It uses the exact same `efficientdet_lite0.tflite` model (INT8 Quantized) used in our Android APK.

In [ ]:
!pip install -q mediapipe opencv-python matplotlib urllib3

In [ ]:
import cv2
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# 1. Download the EfficientDet-Lite0 model (Same INT8 model as Android app)
model_url = 'https://storage.googleapis.com/mediapipe-models/object_detector/efficientdet_lite0/int8/1/efficientdet_lite0.tflite'
urllib.request.urlretrieve(model_url, 'efficientdet_lite0.tflite')

# 2. Download a sample image (A street scene with cars and people)
image_url = 'https://images.unsplash.com/photo-1517737812598-1a43d0ef261c?ixlib=rb-4.0.3&auto=format&fit=crop&w=800&q=80'
urllib.request.urlretrieve(image_url, 'sample.jpg')

print("Model and sample image downloaded successfully.")

In [ ]:
# 3. Initialize the Object Detector
base_options = python.BaseOptions(model_asset_path='efficientdet_lite0.tflite')
options = vision.ObjectDetectorOptions(base_options=base_options, score_threshold=0.3)
detector = vision.ObjectDetector.create_from_options(options)

# 4. Run Inference (This mimics what happens 30x a second on the Android camera feed)
image = mp.Image.create_from_file('sample.jpg')
detection_result = detector.detect(image)

# 5. Visualize the results (Bounding boxes trigger the haptic feedback in the app)
image_copy = np.copy(image.numpy_view())
for detection in detection_result.detections:
    bbox = detection.bounding_box
    start_point = bbox.origin_x, bbox.origin_y
    end_point = bbox.origin_x + bbox.width, bbox.origin_y + bbox.height
    cv2.rectangle(image_copy, start_point, end_point, (0, 255, 0), 3)
    
    category = detection.categories[0]
    category_name = category.category_name
    probability = round(category.score, 2)
    result_text = category_name + ' (' + str(probability) + ')'
    text_location = (bbox.origin_x, bbox.origin_y - 10)
    cv2.putText(image_copy, result_text, text_location, cv2.FONT_HERSHEY_PLAIN, 1.5, (0, 255, 0), 2)

plt.figure(figsize=(10, 10))
plt.imshow(image_copy)
plt.axis('off')
plt.title('AccessApp: On-Device Obstacle Radar Verification')
plt.show()